# 04 · Pricing GLMs: Frequency, Severity and Total Loss

Notebook 03 compared BEV and ICE one factor at a time. Here we fit three GLMs that control for all rating factors at once, and turn the BEV coefficients into tariff relativities.

| Model | Response | Family / link | Relativity |
|---|---|---|---|
| 1. Frequency | own-damage claim count per policy | Poisson / log, offset $\log(\text{exposure})$ | rate ratio $e^{\beta}$ |
| 2. Severity | own-damage cost per claim | Gamma / log | cost ratio $e^{\beta}$ |
| 3. Total loss | 1 if the claim is a total loss | Binomial / logit | odds ratio $e^{\beta}$ |

**Data rules** (same as notebooks 02–03): claims are joined many-to-one on `Policy_ID`, must fall inside the policy period, and the 2025-01-19 storm is excluded. An own-damage claim has `Own_Damage_Amount > 0`. `Powertrain` has ICE as the reference level. Hybrid is kept as its own level, so the tariff covers every vehicle. Rows with a missing rating variable are dropped by `glm()`.

## 1. Setup and Data

In [1]:
suppressWarnings(suppressPackageStartupMessages({
  library(tidyverse)
  library(scales)
  library(broom)
}))

fmt_p <- function(p) if_else(p < 0.001, "< 0.001", number(p, accuracy = 0.001))
p_str <- function(p) if_else(p < 0.001, "p < 0.001", paste("p =", number(p, accuracy = 0.001)))

# Path to a file in data/processed/; works from notebooks/ or the repository root
processed_path <- function(file) {
  path <- file.path(c("../data/processed", "data/processed"), file) |> keep(file.exists) |> first()
  if (is.na(path)) stop("Cannot find ", file, " (run src/01_clean_datasets.R first)")
  path
}

policies <- read_rds(processed_path("policies_cleaned.rds"))            # Powertrain levels: ICE (base), Hybrid, BEV

# Own-damage claims inside the policy period, storm excluded (join checked in src/02_verify_join.R)
od_claims <- read_rds(processed_path("claims_cleaned.rds")) |>
  filter(Own_Damage_Amount > 0, !is_cat_event) |>
  inner_join(select(policies, Policy_ID, Start_Date, End_Date, Powertrain, Vehicle_Value, Vehicle_Age,
                    Driver_Age, District),
             by = "Policy_ID", relationship = "many-to-one") |>
  filter(between(Accident_Date, Start_Date, End_Date))

# Model 1: claim count per policy (0 if none), exposure in vehicle-years
freq_data <- policies |>
  left_join(count(od_claims, Policy_ID, name = "Claim_Count"), by = "Policy_ID") |>
  mutate(Claim_Count = coalesce(Claim_Count, 0L))

# Model 2: drop the amounts above 2x vehicle value (entry errors)
sev_data <- od_claims |> filter(Own_Damage_Amount <= 2 * Vehicle_Value)

# Model 3: theft excluded (a stolen car is written off whatever its powertrain)
tl_data <- od_claims |> filter(Claim_Type != "Theft")

tibble(model = c("1. Frequency (policies)", "2. Severity (claims)", "3. Total loss (claims)"),
       rows  = c(nrow(freq_data), nrow(sev_data), nrow(tl_data)))

model,rows
<chr>,<int>
1. Frequency (policies),38309
2. Severity (claims),5520
3. Total loss (claims),5361


In [2]:
# Relativity table: exp(beta) with a 95% Wald CI. For log(Vehicle_Value) we show 2^beta,
# i.e. the factor when the vehicle value doubles.
relativities <- function(fit) {
  tidy(fit) |>
    filter(term != "(Intercept)") |>
    mutate(base = if_else(term == "log(Vehicle_Value)", 2, exp(1)),
           relativity = base^estimate,
           ci_low  = base^(estimate - 1.96 * std.error),
           ci_high = base^(estimate + 1.96 * std.error)) |>
    transmute(term = if_else(term == "log(Vehicle_Value)", "Vehicle_Value (per doubling)", term),
              beta = number(estimate, 0.001), relativity = number(relativity, 0.001),
              ci_95 = paste0(number(ci_low, 0.001), " – ", number(ci_high, 0.001)),
              p_value = fmt_p(p.value))
}

# BEV coefficient of a fitted model: estimate, standard error, p-value
bev <- function(fit) tidy(fit) |> filter(term == "PowertrainBEV")

## 2. Model 1: Claim Frequency (Poisson GLM)
$$\log E[N_i] = \log(\text{exposure}_i) + \beta_0 + \beta_{\text{Hybrid}} + \beta_{\text{BEV}} + \beta_v \log(\text{Value}_i) + \beta_a \,\text{VehAge}_i + \beta_d \,\text{DriverAge}_i$$

In [3]:
freq_fit <- glm(Claim_Count ~ Powertrain + log(Vehicle_Value) + Vehicle_Age + Driver_Age,
                offset = log(exposure), family = poisson(link = "log"), data = freq_data)

relativities(freq_fit)

dispersion <- sum(residuals(freq_fit, type = "pearson")^2) / df.residual(freq_fit)
b1 <- bev(freq_fit)
b1_qp <- bev(update(freq_fit, family = quasipoisson(link = "log")))   # same fit, SEs scaled by the dispersion
cat("Policies used:", nobs(freq_fit), "| Pearson dispersion:", number(dispersion, 0.001),
    "| BEV p-value with quasi-Poisson SEs:", fmt_p(b1_qp$p.value), "\n")
cat("Takeaway: BEV frequency relativity =", number(exp(b1$estimate), 0.001), paste0("(", p_str(b1$p.value), ")"),
    if_else(b1$p.value < 0.05, "-> differs significantly from 1.00", "-> not significantly different from 1.00"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainHybrid,0.078,1.081,1.010 – 1.157,0.025
PowertrainBEV,0.098,1.103,1.005 – 1.211,0.039
Vehicle_Value (per doubling),-0.154,0.899,0.854 – 0.946,< 0.001
Vehicle_Age,-0.013,0.987,0.977 – 0.997,0.014
Driver_Age,-0.016,0.985,0.983 – 0.986,< 0.001


Policies used: 37338 | Pearson dispersion: 1.104 | BEV p-value with quasi-Poisson SEs: 0.049 


Takeaway: BEV frequency relativity = 1.103 (p = 0.039) -> differs significantly from 1.00 


**Result:**
- **BEV:** relativity **1.10** (95% CI 1.01–1.21, $p = 0.039$).
- **Overdispersion:** the Pearson dispersion is 1.10, so the Poisson standard errors are slightly too small. With quasi-Poisson standard errors the BEV p-value is 0.049, still just below 0.05.
- **Hybrid:** relativity 1.08 ($p = 0.025$).
- **Why this is higher than notebook 03 (RR 1.05, not significant):** more expensive cars have *fewer* claims (×0.90 per doubling of value). Once value is held fixed, the BEV frequency comes out about 10% higher than ICE.

**Takeaway:** at equal value, vehicle age and driver age, BEVs claim about 10% more often. The effect is borderline significant.

## 3. Model 2: Own-Damage Severity (Gamma GLM)
$$\log E[X_j] = \beta_0 + \beta_{\text{Hybrid}} + \beta_{\text{BEV}} + \beta_v \log(\text{Value}_j) + \beta_a \,\text{VehAge}_j + \gamma_{\text{District}}$$
The Gamma family fits right-skewed, positive costs with a constant coefficient of variation; with the log link every coefficient is a multiplicative cost factor.

In [4]:
sev_fit <- glm(Own_Damage_Amount ~ Powertrain + log(Vehicle_Value) + Vehicle_Age + District,
               family = Gamma(link = "log"), data = sev_data)

relativities(sev_fit)

b2 <- bev(sev_fit)
cat("Claims used:", nobs(sev_fit), "\n")
cat("Takeaway: BEV severity relativity =", number(exp(b2$estimate), 0.001),
    paste0("(95% CI ", number(exp(b2$estimate - 1.96 * b2$std.error), 0.001), " – ",
           number(exp(b2$estimate + 1.96 * b2$std.error), 0.001), ", ", p_str(b2$p.value), ")"),
    if_else(b2$p.value < 0.05, "-> significant", "-> not significant"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainHybrid,0.157,1.170,1.065 – 1.284,0.001
PowertrainBEV,0.132,1.141,1.005 – 1.296,0.042
Vehicle_Value (per doubling),0.746,1.677,1.567 – 1.794,< 0.001
Vehicle_Age,0.013,1.013,0.999 – 1.027,0.063
DistrictHaifa,-0.083,0.920,0.811 – 1.044,0.198
DistrictJerusalem,-0.077,0.926,0.819 – 1.047,0.219
DistrictJudea & Samaria,-0.324,0.723,0.522 – 1.002,0.051
DistrictNorth,0.016,1.016,0.893 – 1.157,0.805
DistrictSouth,0.066,1.069,0.950 – 1.202,0.269


Claims used: 5520 


Takeaway: BEV severity relativity = 1.141 (95% CI 1.005 – 1.296, p = 0.042) -> significant 


**Result:**
- **BEV:** relativity **1.14** (95% CI 1.01–1.30, $p = 0.042$).
- **Hybrid:** relativity 1.17 ($p = 0.001$).
- **Vehicle value:** the strongest cost driver. A car worth twice as much has claims about 1.68× as expensive, an elasticity of 0.75.
- **District:** no district differs significantly from Center.
- **Why this is below notebook 03's 1.29:** that figure was a ratio of *geometric* means inside the age ≤ 5 cohort with three value tiers. The Gamma GLM models the *mean* cost on all ages with continuous value.

**Takeaway:** at equal value, age and district, a BEV own-damage claim costs about 14% more than an ICE claim.

## 4. Model 3: Total-Loss Probability (Logistic GLM)
$$\text{logit}\, P(\text{TL}_j = 1) = \beta_0 + \beta_{\text{Hybrid}} + \beta_{\text{BEV}} + \beta_v \log(\text{Value}_j) + \beta_a \,\text{VehAge}_j$$

In [5]:
tl_fit <- glm(is_total_loss_clean ~ Powertrain + log(Vehicle_Value) + Vehicle_Age,
              family = binomial(link = "logit"), data = tl_data)

relativities(tl_fit)

b3 <- bev(tl_fit)
cat("Claims used:", nobs(tl_fit), "| total losses:", sum(tl_fit$y), "\n")
cat("Takeaway: BEV total-loss odds ratio =", number(exp(b3$estimate), 0.001),
    paste0("(95% CI ", number(exp(b3$estimate - 1.96 * b3$std.error), 0.001), " – ",
           number(exp(b3$estimate + 1.96 * b3$std.error), 0.001), ", ", p_str(b3$p.value), ")"),
    if_else(b3$p.value < 0.05, "-> significant", "-> not significant"), "\n")

term,beta,relativity,ci_95,p_value
<chr>,<chr>,<chr>,<chr>,<chr>
PowertrainHybrid,0.036,1.037,0.781 – 1.376,0.803
PowertrainBEV,0.665,1.945,1.385 – 2.731,< 0.001
Vehicle_Value (per doubling),-0.448,0.733,0.608 – 0.883,0.001
Vehicle_Age,0.029,1.030,0.995 – 1.066,0.094


Claims used: 5307 | total losses: 411 


Takeaway: BEV total-loss odds ratio = 1.945 (95% CI 1.385 – 2.731, p < 0.001) -> significant 


**Result:**
- **BEV:** odds ratio **1.95** (95% CI 1.39–2.73, $p < 0.001$).
- **Hybrid:** 1.04, not significant.
- **Vehicle value:** higher-value cars are written off less often (×0.73 per doubling of value), because repairs are a smaller share of a more valuable car.
- **Why this is clearer than the crude tests:** BEVs are newer and pricier, which hides their higher write-off rate in a raw comparison (notebook 03).

**Takeaway:** at equal value and age, a damaged BEV has about twice the odds of being written off, which supports the battery-damage argument.

## 5. Tariff Formula: Pure Premium Relativity
With a multiplicative tariff, the pure premium is frequency × severity, so the powertrain relativities multiply:
$$\text{Pure Premium Relativity}_{\text{BEV}} = \text{Relativity}_{\text{Freq}} \times \text{Relativity}_{\text{Sev}} = e^{\beta^{\text{freq}}_{\text{BEV}} + \beta^{\text{sev}}_{\text{BEV}}}$$
Treating the two models as independent, the 95% CI uses $\text{se} = \sqrt{\text{se}_{\text{freq}}^2 + \text{se}_{\text{sev}}^2}$ on the log scale.

Total losses are already part of the own-damage cost in Model 2, so the total-loss model is **not** multiplied in again. That would count the same losses twice.

In [6]:
pure_premium <- map(c("Hybrid", "BEV"), \(pt) {
  f <- tidy(freq_fit) |> filter(term == paste0("Powertrain", pt))
  s <- tidy(sev_fit)  |> filter(term == paste0("Powertrain", pt))
  se <- sqrt(f$std.error^2 + s$std.error^2)
  tibble(Powertrain = pt,
         frequency = exp(f$estimate), severity = exp(s$estimate),
         pure_premium = exp(f$estimate + s$estimate),
         ci_low  = exp(f$estimate + s$estimate - 1.96 * se),
         ci_high = exp(f$estimate + s$estimate + 1.96 * se))
}) |> list_rbind()

pure_premium |>
  mutate(across(frequency:pure_premium, ~ number(.x, 0.001)),
         ci_95 = paste0(number(ci_low, 0.001), " – ", number(ci_high, 0.001)), .keep = "unused")

Powertrain,frequency,severity,pure_premium,ci_95
<chr>,<chr>,<chr>,<chr>,<chr>
Hybrid,1.081,1.170,1.264,1.126 – 1.419
BEV,1.103,1.141,1.259,1.075 – 1.474


**Result:**
$$\text{PP}_{\text{BEV}} = 1.103 \times 1.141 = \mathbf{1.26} \quad (95\%\text{ CI } 1.08\text{–}1.47)$$
- **BEV:** the own-damage pure premium is about 26% above an ICE car of the same value and age.
  - About 10 percentage points come from frequency and about 14 from severity.
  - The combined factor is clearly significant, even though each part on its own is only just below 5%.
- **Hybrid:** almost the same factor, 1.08 × 1.17 = 1.26.

**Takeaway:** the tariff should carry a powertrain relativity of about 1.26 for BEVs on the own-damage premium, on top of the vehicle-value factor.

## 6. Pricing and Underwriting Recommendations

| Component | Model | BEV relativity / effect | Tariff impact | Underwriting recommendation |
|---|---|---|---|---|
| Frequency | Poisson GLM | 1.10 (1.01–1.21), $p = 0.039$ ($0.049$ quasi-Poisson) | Part of the combined BEV factor; borderline on its own | Monitor as BEV exposure grows; re-estimate each year |
| Own-damage severity | Gamma GLM | 1.14 (1.01–1.30), $p = 0.042$ | Part of the combined BEV factor | Repair-network agreements for BEVs; review deductibles |
| **Own-damage pure premium** | Frequency × severity | **1.26** (1.08–1.47) | **BEV own-damage relativity ≈ 1.26**, on top of vehicle value | Main pricing change |
| Total loss | Logistic GLM | OR 1.95 (1.39–2.73), $p < 0.001$ | **No separate loading**: total losses are already in the severity model, so a second loading would double count | Battery inspection after impacts; specialist battery-repair and salvage partners to avoid write-offs |
| Third-party severity | t-test (notebook 03) | 0.95, $p = 0.42$ | 1.00: no change | Keep the third-party rate for BEVs the same as for ICE |

**Final tariff (own-damage part, multiplicative):**
$$\text{Premium} = \text{Base} \times f(\text{Vehicle Value}) \times f(\text{Vehicle Age}) \times f(\text{Driver Age}) \times \underbrace{1.26}_{\text{BEV}}$$

**Limitations:**
- About 4,000 BEV vehicle-years and 631 BEV claims, so the frequency and severity factors are each only just significant. The combined factor is the more reliable number.
- No large-loss cap is applied yet: the audit suggests capping own damage at P99.5 as a sensitivity check.
- The storm is excluded, so a separate CAT loading is needed. The storm barely changes the BEV/ICE frequency ratio (1.057 with it, 1.059 without; data quality audit §6).